In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    temperature=0,
    groq_api_key="Your API key ",
    model_name="openai/gpt-oss-120b"
)

response = llm.invoke("The first person to land on the moon was...")
print(response.content)

The first person to set foot on the Moon was **Neil Armstrong**, who did so on July 20 1969 during NASA’s Apollo 11 mission. He famously described the moment as “one small step for [a] man, one giant leap for mankind.”


In [2]:
# Extract all key information from the job posting
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    "https://careers.nike.com/nike-store-associate-athlete-la-roca-del-valles-bcn-contrato-permanente-part-time-20hs/job/R-94634"
)

page_data = loader.load().pop().page_content

print(page_data[:2000])

C:\Users\Siddhi\AppData\Local\Temp\ipykernel_11296\2758769230.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.






















Nike Store Associate (Athlete) - La Roca del Valles, BCN - Contrato Permanente - Part Time (20hs)














































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Convers

In [3]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE:
    {page_data}

    ### INSTRUCTION:
    The scraped text is from the career's page of a website.
    Your job is to extract the job postings and return them in JSON format containing the
    following keys: 'role', 'experience', 'skills' and 'description'.
    Only return the valid JSON.

    ### VALID JSON (NO PREAMBLE):
    """
)

chain_extract = prompt_extract | llm

res = chain_extract.invoke(input={'page_data': page_data})

print(res.content)

{
  "role": "Nike Store Associate (Athlete)",
  "experience": "Previous retail or sales experience is a plus; no specific years required.",
  "skills": [
    "Passion for Nike and sport",
    "Strong customer service and communication skills",
    "Product knowledge of Nike merchandise",
    "Ability to work in a team and support coaches/leads",
    "Proactive, open‑minded and motivated attitude",
    "Flexibility to work various shifts, including weekends",
    "Basic digital skills to connect online and offline sales",
    "Ability to handle cash register, inventory, and visual merchandising"
  ],
  "description": "As a Nike Store Associate (Athlete) you will represent the brand in the La Roca del Valles store in Barcelona. You will use your product knowledge and exceptional customer service to deliver premium brand experiences, support new staff training, work across store departments (cash register, inventory, replenishment, visual displays), recommend products to meet sales target

In [4]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Nike Store Associate (Athlete)',
 'experience': 'Previous retail or sales experience is a plus; no specific years required.',
 'skills': ['Passion for Nike and sport',
  'Strong customer service and communication skills',
  'Product knowledge of Nike merchandise',
  'Ability to work in a team and support coaches/leads',
  'Proactive, open‑minded and motivated attitude',
  'Flexibility to work various shifts, including weekends',
  'Basic digital skills to connect online and offline sales',
  'Ability to handle cash register, inventory, and visual merchandising'],
 'description': 'As a Nike Store Associate (Athlete) you will represent the brand in the La Roca del Valles store in Barcelona. You will use your product knowledge and exceptional customer service to deliver premium brand experiences, support new staff training, work across store departments (cash register, inventory, replenishment, visual displays), recommend products to meet sales targets, and leverage digital tool

In [5]:
type(json_res)

dict

In [6]:
import pandas as pd
df= pd.read_csv("techstack_active_links.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://react.dev/
1,"Angular, .NET, SQL Server",https://dotnet.microsoft.com/
2,"Vue.js, Ruby on Rails, PostgreSQL",https://vuejs.org/
3,"Python, Django, MySQL",https://www.python.org/
4,"Java, Spring Boot, Oracle",https://www.oracle.com/java/
5,"Flutter, Firebase, GraphQL",https://flutter.dev/
6,"WordPress, PHP, MySQL",https://wordpress.org/
7,"Magento, PHP, MySQL",https://business.adobe.com/products/commerce.html
8,"React Native, Node.js, MongoDB",https://reactnative.dev/
9,"iOS, Swift, Core Data",https://developer.apple.com/ios/


In [7]:
import uuid
import chromadb

client = chromadb.PersistentClient("vectorestore")

collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(
            documents=[row["Techstack"]],
            metadatas={"links": row["Links"]},
            ids=[str(uuid.uuid4())]
        )



In [11]:
results = collection.query(
    query_texts=["Experience in Python", "Expertise in React Native"],
    n_results=2
)

links = results.get("metadatas")

print(links)

[[{'links': 'https://www.tensorflow.org/'}, {'links': 'https://www.python.org/'}], [{'links': 'https://reactnative.dev/'}, {'links': 'https://react.dev/'}]]


In [12]:
job=json_res
job['skills']

['Passion for Nike and sport',
 'Strong customer service and communication skills',
 'Product knowledge of Nike merchandise',
 'Ability to work in a team and support coaches/leads',
 'Proactive, open‑minded and motivated attitude',
 'Flexibility to work various shifts, including weekends',
 'Basic digital skills to connect online and offline sales',
 'Ability to handle cash register, inventory, and visual merchandising']

In [13]:
prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:
    {job_description}

    ### INSTRUCTION:
    You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to the seamless integration of business processes through automated tools.

    Over our experience, we have empowered numerous enterprises with tailored solutions, fostering process optimization, cost reduction, and heightened overall efficiency.

    Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ in fulfilling their needs.

    Also add the most relevant ones from the following links to showcase AtliQ's portfolio: {link_list}

    Remember you are Mohan, BOE at AtliQ.

    Do not provide a preamble.

    ### EMAIL (NO PREAMBLE):

    """
)

chain_email = prompt_email | llm

res = chain_email.invoke({
    "job_description": str(job),
    "link_list": links
})

print(res.content)

**Subject:** Boost Nike Barcelona Store Performance with AI‑Driven Retail Solutions  

Hi [Hiring Manager’s Name],

I’m Mohan, Business Development Executive at **AtliQ**, an AI‑focused software consulting firm that helps leading brands turn their retail operations into data‑driven profit engines.  

I noticed the opening for a **Nike Store Associate (Athlete)** at La Roca del Valles, and I wanted to share how AtliQ can empower your team to deliver the premium, omnichannel experience Nike promises—while reducing operational overhead and driving sales.

---

### Why AtliQ is the Ideal Partner for Nike Barcelona  

| Nike Need | AtliQ Solution | Business Impact |
|-----------|----------------|-----------------|
| **Seamless online‑offline integration** | Custom **React** web dashboards + **React Native** mobile apps that sync POS, inventory, and e‑commerce in real time. | Faster replenishment, zero‑stock outs, unified customer view. |
| **Data‑backed product recommendations** | AI models

In [14]:
from langchain_core.prompts import PromptTemplate

prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:
    {job_description}

    ### ATLIQ PORTFOLIO:
    {link_list}

    ### INSTRUCTIONS:

    You are Mohan, a Business Development Executive at AtliQ,
    an AI and Software Consulting company.

    Write a professional and personalized cold email to the
    company based on the job description provided above.

    The email should:

    1. Have a clear and professional subject line.
    2. Address the hiring manager or company professionally.
    3. Briefly mention the relevant job opening.
    4. Explain how AtliQ can help the company based on the
       requirements mentioned in the job description.
    5. Mention only the technologies, services, and capabilities
       that are relevant to the job description.
    6. Include the most relevant portfolio links from the
       provided AtliQ portfolio.
    7. Keep the email concise, professional, and easy to read.
    8. End with a clear call to action for a short discussion.
    9. Do not invent statistics, clients, results, revenue,
       percentages, or case studies.
    10. Do not mention information that is not available in the
        job description or portfolio links.
    11. Do not provide any explanation before or after the email.
    12. Return only the final email.

    ### EMAIL FORMAT:

    Subject: [Professional subject]

    Hi [Hiring Manager/Team],

    [Personalized opening based on the job description.]

    [Explain how AtliQ's relevant capabilities can help.]

    [Mention 1-2 relevant portfolio links.]

    [Short call to action.]

    Best regards,
    Mohan
    Business Development Executive
    AtliQ
    """
)

chain_email = prompt_email | llm

res = chain_email.invoke({
    "job_description": str(job),
    "link_list": str(links)
})

print(res.content)

Subject: Boosting Nike Store Performance with AI‑Driven Retail Solutions  

Hi Hiring Team,

I was excited to see the Nike Store Associate (Athlete) opening for the La Roca del Valles location. Your focus on delivering premium brand experiences, seamless online‑offline integration, and efficient inventory and visual merchandising aligns perfectly with the digital transformation services AtliQ provides to retail brands.

At AtliQ, we help retailers elevate customer service and operational efficiency through AI‑powered tools built on TensorFlow and Python, and intuitive front‑end experiences using React/React Native. Our solutions include:

* **Omnichannel sales integration** – real‑time product recommendations and mobile checkout that connect online browsing with in‑store purchases.  
* **Smart inventory & replenishment** – predictive analytics to keep shelves stocked and reduce out‑of‑stock situations.  
* **Digital visual merchandising** – interactive displays and AR‑enabled product s